# Actividad: Mapas Coropléticos de Migración en la Región Metropolitana

**Curso: Análisis de Datos Espaciales con Python**  


---

## Contexto

En la clase anterior aprendimos a construir **mapas coropléticos** usando datos del PIB per cápita de México: cómo clasificar una variable continua en categorías discretas y pintar cada polígono según su clase.

Hoy vamos a **aplicar esas mismas técnicas** a un problema real y cercano: la **migración internacional en la Región Metropolitana de Chile**, usando datos del **Censo de Población y Vivienda 2024** publicados por el INE.

### Objetivos de aprendizaje

1. Practicar la construcción de mapas coropléticos con datos reales chilenos.
2. Comparar al menos 3 clasificadores (Intervalos Iguales, Cuantiles, Fisher-Jenks) sobre una misma variable.
3. Analizar críticamente qué revela (y qué oculta) cada clasificación sobre los patrones de migración.
4. Interpretar la distribución espacial de la inmigración a nivel comunal en Santiago.

### Prerrequisito

Antes de comenzar debes tener listo el archivo `migrant_reg_counts.csv`. Tienes dos opciones:

- **Opción A:** Ejecutar primero el notebook `03_actividad3_preambulo.ipynb`, que procesa los microdatos del Censo 2024 y genera ese archivo.
- **Opción B:** Usar directamente el archivo `migrant_reg_counts.csv` si ya fue generado previamente (está en `datos/external/censo2024/personas/`).


---
## PARTE 1: Preparación del entorno y datos (15 minutos)
### 1.1 Importar librerías


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import mapclassify
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["figure.dpi"] = 96

### 1.2 Cargar los datos de migración

Cargamos el CSV generado por el preámbulo. Este archivo contiene el número de migrantes por nacionalidad/continente y comuna en la RM.


In [ ]:
migrant_reg_counts = pd.read_csv('datos/external/censo2024/personas/migrant_reg_counts.csv')


migrant_reg_counts.head(10)

### 1.3 Agregar: total de migrantes por comuna

El archivo tiene el detalle por país/continente de origen. Para hacer mapas coropléticos necesitamos **un valor por comuna**, así que vamos a sumar el total de migrantes en cada una.

**Ejecuta la celda y observa:** ¿Qué comuna tiene más migrantes en términos absolutos? 


### 1.4 Cargar la geometría comunal

Usamos el shapefile de comunas de la RM del Censo 2017 (misma geometría comunal).


In [ ]:
comunas_gdf = gpd.read_file("datos/external/censo2017/R13/COMUNA_C17.shp")


comunas_gdf.head()

### 1.5 Unir datos de migración con la geometría

Hacemos un **merge** entre la geometría y nuestros datos. 


Notar que los nombres de las comunas varian entre ambos datasets. 

In [ ]:
import unicodedata

def normalizar(texto):
    """Quita tildes y convierte a mayúsculas para hacer match."""
    if pd.isna(texto):
        return texto
    texto = str(texto).strip().upper()
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    return texto



### 1.6 Calcular el porcentaje de migrantes

Para hacer mapas coropléticos comparables entre comunas de distintos tamaños, es mejor usar una **tasa** (porcentaje) en vez del número absoluto.

Calcula el porcentaje de inmigrantes por comuna.


In [ ]:
poblacion_rm = {
    "SANTIAGO": 404000, "CERRILLOS": 88000, "CERRO NAVIA": 132000,
    "CONCHALI": 128000, "EL BOSQUE": 162000, "ESTACION CENTRAL": 178000,
    "HUECHURABA": 112000, "INDEPENDENCIA": 115000, "LA CISTERNA": 95000,
    "LA FLORIDA": 366000, "LA GRANJA": 116000, "LA PINTANA": 177000,
    "LA REINA": 88000, "LAS CONDES": 249000, "LO BARNECHEA": 124000,
    "LO ESPEJO": 96000, "LO PRADO": 94000, "MACUL": 116000,
    "MAIPU": 578000, "NUNOA": 228000, "PEDRO AGUIRRE CERDA": 95000,
    "PENALOLEN": 252000, "PROVIDENCIA": 142000, "PUDAHUEL": 265000,
    "QUILICURA": 255000, "QUINTA NORMAL": 128000, "RECOLETA": 175000,
    "RENCA": 158000, "SAN JOAQUIN": 94000, "SAN MIGUEL": 107000,
    "SAN RAMON": 82000, "VITACURA": 87000, "PUENTE ALTO": 648000,
    "SAN BERNARDO": 330000, "COLINA": 195000, "LAMPA": 130000,
    "PADRE HURTADO": 68000, "PENAFLOR": 80000, "PIRQUE": 26000,
    "TALAGANTE": 72000, "BUIN": 95000, "CALERA DE TANGO": 30000,
    "PAINE": 82000, "SAN JOSE DE MAIPO": 18000, "MELIPILLA": 110000,
    "ALHUE": 6000, "CURACAVI": 38000, "MARIA PINTO": 12000,
    "SAN PEDRO": 4000, "TILTIL": 22000, "ISLA DE MAIPO": 35000
}


---
## PARTE 2: Exploración estadística (10 minutos)



### 2.1 Distribución de la variable

Explora los datos con ayuda de un histograma, un boxplot y con describe().


Cual es la comuna con mayor porcentaje de migrantes? Y la que tiene menor porcentaje? ¿Cuál es la mediana y media del porcentaje de migrantes en las comunas de la RM? ¿Qué nos dice esto sobre la distribución de migrantes en la región?

### Pregunta para reflexionar

Observa la distribución: ¿Es simétrica o asimétrica? ¿Tiene outliers? 


## PARTE 3: Mapas coropléticos con distintos clasificadores (20 minutos)



### 3.1 Primer vistazo: el mapa sin datos

Visualiza la geometria de la RM

Ahora crea un mapa de coropletas para visualizar el porcentaje de migrantes por comuna en la Región Metropolitana.

### 3.2 Intervalos Iguales (k=5)

Recuerda: divide el rango total en **k partes de igual ancho**.


### 3.3 Cuantiles (k=5)

Recuerda: cada clase tiene **aproximadamente el mismo número de comunas**.


### 3.4 Fisher-Jenks (k=5)

Recuerda: **minimiza la varianza intra-clase** usando programación dinámica. Es la clasificación **óptima** estadísticamente.


### 3.5 Comparación lado a lado

Observa los tres mapas juntos. ¿Hay comunas cambian de color entre un clasificador y otro?


### 3.6 Comparación numérica: ADCM

¿Cuál tiene el menor ADCM? ¿Coincide con el mapa que visualmente te parece más informativo?


---
## PARTE 4: Análisis e interpretación (15 minutos)

### 4.1 Heatmap comparativo

Observa en qué comunas los tres clasificadores coinciden y en cuáles discrepan.


### 4.2 Preguntas de análisis

**Pregunta 1:** ¿Qué comunas concentran la mayor proporción de migrantes? ¿Coincide con lo que conoces de Santiago? ¿Por qué crees que estas comunas atraen más migración?


**Pregunta 2:** Compara los tres mapas (Intervalos Iguales, Cuantiles, Fisher-Jenks). ¿En qué comunas los clasificadores "discrepan" más, es decir, las asignan a clases muy diferentes? ¿Por qué ocurre esto?


**Pregunta 3:** Si tuvieras que presentar estos datos en un informe para una política pública sobre integración de migrantes, ¿qué clasificador elegirías y por qué? 



### Referencias

- Clase 5: Mapas Coropléticos (basada en Rey, Arribas-Bel y Wolf, 2020. *Geographic Data Science with Python*).
- Datos: [Censo 2024 - INE Chile](https://censo2024.ine.gob.cl/)
- Procesamiento original: [daniopitz/censo2024](https://github.com/daniopitz/censo2024)
